# Week 4 · MLP for ETA: train, regularize, and beat the Week 3 baseline

# Requirements: pip install torch scikit-learn pandas numpy matplotlib
# Uses: the `zoro` package. Weeks 1-4 are CPU-only.

We train a neural net to predict **`delay_hours`**, the same target, features, and
time-aware split as Week 3, then do the thing that defines AI engineering: compare it
against the gradient-boosting baseline on the test set, and run a first **error
analysis** by carrier and lane.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from zoro import data
print("torch:", torch.__version__)

### Same data, same split: the comparison is only honest if nothing else moved

A neural net does not get a fresh advantage: identical features, identical cut dates,
identical seed. The only thing that changes is the model family.

In [ ]:
# Build the feature table (same as Week 3): join, impute planted NaNs, derive calendar features.
def build_features(n=100_000, seed=42):
    ships = data.shipments(n, seed=seed)
    lanes = data.lanes(20, seed=11)
    carriers = data.carriers(20, seed=7)
    df = (ships
          .merge(lanes[["lane_id", "distance_km"]], on="lane_id", how="left")
          .merge(carriers[["carrier_id", "on_time_rate", "fleet_size", "region"]], on="carrier_id", how="left"))
    df["weight_kg"] = df["weight_kg"].fillna(df.groupby("commodity")["weight_kg"].transform("median"))
    df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
    df = df.drop_duplicates().reset_index(drop=True)
    df["month"] = df["planned_departure"].dt.month
    df["day_of_week"] = df["planned_departure"].dt.dayofweek
    return df

df = build_features(100_000, seed=42)

NUM_COLS = ["distance_km", "weight_kg", "value_usd", "on_time_rate", "fleet_size", "month", "day_of_week"]
CAT_COLS = ["weather_severity", "commodity"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
prep = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
])
print("feature table:", df.shape)

In [ ]:
# Same time-aware split as Week 3: chronological, never shuffled.
CUT1 = "2025-08-01"
CUT2 = "2025-10-01"
train_df = df[df["planned_departure"] < CUT1].reset_index(drop=True)
val_df   = df[(df["planned_departure"] >= CUT1) & (df["planned_departure"] < CUT2)].reset_index(drop=True)
test_df  = df[df["planned_departure"] >= CUT2].reset_index(drop=True)
print("train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))

### Scale, then wrap in tensors and a DataLoader

Neural nets want standardized inputs (fit the scaler on **train only**), a float tensor
matrix, and **mini-batches**, `DataLoader` shuffles the training data and hands out
batches. Shuffle train; never shuffle val/test order.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

y_train = train_df["delay_hours"].to_numpy(np.float32).reshape(-1, 1)
y_val   = val_df["delay_hours"].to_numpy(np.float32).reshape(-1, 1)
y_test  = test_df["delay_hours"].to_numpy(np.float32).reshape(-1, 1)

X_train = prep.fit_transform(train_df).astype(np.float32)
X_val   = prep.transform(val_df).astype(np.float32)
X_test  = prep.transform(test_df).astype(np.float32)
print("X_train:", X_train.shape, "| n features:", X_train.shape[1])

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds   = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds  = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=1024)
print("DataLoader ready: train batches =", len(train_dl), "| val batches =", len(val_dl))

### The model and the training loop

`nn.Module` holds the layers; **dropout** (train-only) and **weight decay** (via the
`AdamW` optimizer) are the two regularizers that fight overfitting. The loop is the same
five lines, wrapped in a per-epoch pass that records both train and validation loss.

In [ ]:
torch.manual_seed(0)

class EtaMLP(torch.nn.Module):
    def __init__(self, n_in, width=128, dropout=0.1):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_in, width), torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Linear(width, width // 2), torch.nn.ReLU(),
            torch.nn.Linear(width // 2, 1),
        )
    def forward(self, x):
        return self.net(x)

model = EtaMLP(n_in=X_train.shape[1])
loss_fn = torch.nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 20
train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    model.train()
    total, n = 0.0, 0
    for xb, yb in train_dl:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
        n += len(xb)
    train_losses.append(total / n)

    model.eval()
    vt, vn = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            loss = loss_fn(model(xb), yb)
            vt += loss.item() * len(xb)
            vn += len(xb)
    val_losses.append(vt / vn)
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:>3}  train MSE={train_losses[-1]:.4f}  val MSE={val_losses[-1]:.4f}")

### Read the learning curves

Plot train vs validation loss. Read the *gap* between the curves, not just the values:
both high and flat → underfitting; train low and val high with a widening gap →
overfitting; both low and converging → healthy. If you see a widening gap, that is the
dropout/weight-decay knobs asking to be turned.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Learning curves - read the gap between train and val")
plt.legend()
plt.grid(True)
plt.show()

### Evaluate on test, and compare against the Week 3 baseline

Predict the held-out test set in one pass, compute MAE/RMSE, then retrain the Week 3
gradient-boosting baseline **on the same split** for a fair head-to-head. The gate: the
neural net must be *compared against the baseline with the delta stated*, whether it
wins or loses.

In [ ]:
model.eval()
with torch.no_grad():
    preds_test = model(torch.from_numpy(X_test)).numpy().reshape(-1)

from sklearn.metrics import mean_absolute_error, mean_squared_error
nn_mae = mean_absolute_error(y_test.reshape(-1), preds_test)
nn_rmse = np.sqrt(mean_squared_error(y_test.reshape(-1), preds_test))
print(f"Neural MLP:              MAE={nn_mae:.3f}  RMSE={nn_rmse:.3f}")

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
gb = Pipeline([("prep", prep), ("model", GradientBoostingRegressor(n_estimators=100, random_state=0))])
gb.fit(train_df, train_df["delay_hours"])
gb_preds = gb.predict(test_df)
gb_mae = mean_absolute_error(y_test.reshape(-1), gb_preds)
gb_rmse = np.sqrt(mean_squared_error(y_test.reshape(-1), gb_preds))
print(f"GradientBoosting baseline: MAE={gb_mae:.3f}  RMSE={gb_rmse:.3f}")
print(f"delta (neural - baseline): {nn_mae - gb_mae:+.3f} hours MAE")

### Error analysis: where does the model fail?

A metric tells you *how wrong*; error analysis tells you *where to look*. Slice the
neural net's absolute error by **carrier** and **lane**, find the biggest cluster, and
form a hypothesis. That is the habit Ng ranks as the biggest predictor of team
velocity.

In [ ]:
test_meta = test_df[["carrier_id", "lane_id"]].reset_index(drop=True).copy()
test_meta["err"] = np.abs(preds_test - y_test.reshape(-1))

by_carrier = test_meta.groupby("carrier_id")["err"].mean().sort_values(ascending=False).head(5)
by_lane = test_meta.groupby("lane_id")["err"].mean().sort_values(ascending=False).head(5)
print("worst 5 carriers by mean abs error (test):")
print(by_carrier.round(3).to_string())
print()
print("worst 5 lanes by mean abs error (test):")
print(by_lane.round(3).to_string())

worst_lane = by_lane.idxmax()
print()
print("biggest error cluster: lane", worst_lane)
print("hypothesis: a few extreme delays dominate this lane's mean error - check weather/seasonality before blaming the model.")

### The metric

Three numbers to close: the neural test MAE, the baseline test MAE, and the delta
between them.

In [ ]:
print("NEURAL_TEST_MAE:", round(float(nn_mae), 3))
print("BASELINE_TEST_MAE:", round(float(gb_mae), 3))
print("DELTA:", round(float(nn_mae - gb_mae), 3))